# Ethanol: torsional motion at two temperatures
Use **ASE + MACE-OFF23-small** to track the C–C–O–H dihedral in nine atoms.
Compare independent runs at the temperatures in `TEMPERATURES`, starting from the same relaxed geometry.
A torsional transition is not guaranteed; angular fluctuations are already informative.
These short trajectories do not give converged conformer populations or free energies.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from ase import Atoms, units
from ase.build import molecule
from ase.io import read, write
from ase.optimize import BFGS
from ase.md.langevin import Langevin
from ase.md.verlet import VelocityVerlet
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary

sys.path.insert(0, str(Path.cwd().parent.parent.parent))
from demo_tools.helpers import start_exercise
from demo_tools.view_mols import view
_, OUTPUT = start_exercise()
torch.set_num_threads(int(os.environ.get('SLURM_CPUS_PER_TASK', '1')))

from mace.calculators import MACECalculator
calc = MACECalculator(model_paths='../../models/MACE-OFF23_small.model',
                 device='cpu', default_dtype='float64')


## 1. Relax ethanol and identify the torsion
For ASE’s ethanol geometry, C–C–O–H corresponds to indices [0, 1, 2, 3]. These indices are specific to this atom ordering.

In [ ]:
initial = molecule('CH3CH2OH')
initial.info.update(total_charge=0, total_spin=0)
initial.calc = calc
assert BFGS(initial, logfile=str(OUTPUT/'relax.log')).run(fmax=0.02, steps=200)
TORSION = [0,1,2,3]
print(list(enumerate(initial.get_chemical_symbols())))
print('Initial dihedral:',initial.get_dihedral(*TORSION))
view(initial)
#view(initial, highlight_atoms=TORSION)


## 2. Run two short NVT trajectories
Use fresh copies, not sequential heating. Both start from the same geometry, with separately initialized velocities. The thermostat acts on all Cartesian degrees of freedom; we remove net translation only initially.

In [ ]:
DT_FS = 0.5
STEPS = 1000
TEMPERATURES = [100,1000]
trajectories = {}
start = time.perf_counter()
for temperature in TEMPERATURES:
    atoms = initial.copy()
    atoms.calc = calc
    rng = np.random.default_rng(temperature)
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, rng=rng)
    Stationary(atoms)
    path = OUTPUT/f'ethanol_{temperature}K.traj'
    dyn = Langevin(atoms, DT_FS*units.fs, temperature_K=temperature,
                   friction=0.01/units.fs, fixcm=False, rng=rng,
                   trajectory=str(path), loginterval=5)
    dyn.run(STEPS)
    trajectories[temperature] = read(path,index=':')
print(f'Elapsed: {(time.perf_counter()-start)/60:.1f} min')


## 3. Compare torsions
Unwrap the angle to avoid artificial jumps at ±180°. A wider trace suggests larger fluctuations; it does not by itself establish a different equilibrium conformer population.

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(7,5),sharex=True)
for temperature, frames in trajectories.items():
    t = np.arange(len(frames))*5*DT_FS
    angles = np.array([a.get_dihedral(*TORSION) for a in frames])
    continuous = np.rad2deg(np.unwrap(np.deg2rad(angles)))
    ax[0].plot(t,continuous,label=f'{temperature} K')
    ax[1].plot(t,[a.get_temperature() for a in frames],label=f'{temperature} K')
ax[0].set_ylabel('Unwrapped dihedral (degrees)'); ax[0].legend()
ax[1].set(xlabel='Time (fs)',ylabel='Instantaneous T (K)')
plt.tight_layout(); plt.show()
view(trajectories[TEMPERATURES[-1]][::2],animate=True)


In [ ]:
view(trajectories[TEMPERATURES[0]][::2],animate=True)

## Try and explain
Does the higher-temperature run explore a wider angular range? Repeat with another seed.
Is the chosen trajectory length enough to measure a conformer ratio? What would you need to check before
calling an angular excursion a transition between distinct conformers?